# Combined neuron set examples

Examples for how to use different types of combined neuron sets.

In [ ]:
import obi_one as obi
from pathlib import Path

### __Initialization:__ Loading a circuit

In [ ]:
circuit_name = "N_10__top_nodes_dim6"

circuit_path_prefix = Path("../data/tiny_circuits")
matrix_path_prefix = Path("../data/connectivity_matrices")  # OPTIONAL: Connectivity matrix path only required for some of the node sets in this example notebook; can be set to None

circuit_path = circuit_path_prefix / circuit_name / "circuit_config.json"
if matrix_path_prefix is None:
    matrix_path = None
else:
    matrix_path = matrix_path_prefix / circuit_name / "connectivity_matrix.h5"
circuit = obi.Circuit(name=circuit_name, path=str(circuit_path), matrix_path=str(matrix_path))

print(f"Circuit '{circuit}' with {circuit.sonata_circuit.nodes[circuit.default_population_name].size} neurons and {circuit.sonata_circuit.edges[circuit.default_edge_population_name].size} synapses")
print(f"Node populations: '{circuit.sonata_circuit.nodes.population_names}'")
print(f"Edge populations: '{circuit.sonata_circuit.edges.population_names}'")

### **Combined neuron set examples**

Note: The combined neuron sets may span multiple node populations.

In [ ]:
from obi_one.scientific.blocks.neuron_sets.combined import BiophysicalCombinedNeuronSet, SetOperation, VirtualCombinedNeuronSet
from obi_one.scientific.blocks.neuron_sets.population import VirtualPopulationNeuronSet
from obi_one.scientific.blocks.neuron_sets.predefined import BiophysicalPopulationPredefinedNeuronSet
from obi_one.scientific.unions.unions_neuron_sets import BiophysicalNeuronSetReference, VirtualNeuronSetReference


#### (a) BiophysicalCombinedNeuronSet — union, intersection, difference

In [ ]:
# Create two base neuron sets to combine
nset_a = BiophysicalPopulationPredefinedNeuronSet(node_set="All", population="S1nonbarrel_neurons", sample_percentage=50, sample_seed=0)
nset_a.set_block_name("nset_a")

nset_b = BiophysicalPopulationPredefinedNeuronSet(node_set="All", population="S1nonbarrel_neurons", sample_percentage=50, sample_seed=1)
nset_b.set_block_name("nset_b")

# Wrap in BlockReferences (as required by the combined neuron set fields)
ref_a = BiophysicalNeuronSetReference(block_dict_name="neuron_sets", block_name="nset_a")
ref_a.block = nset_a
ref_b = BiophysicalNeuronSetReference(block_dict_name="neuron_sets", block_name="nset_b")
ref_b.block = nset_b

print(f"Set A (50%, seed 0): {nset_a.get_neuron_ids(circuit)}")
print(f"Set B (50%, seed 1): {nset_b.get_neuron_ids(circuit)}")

In [ ]:
# Union
combined_union = BiophysicalCombinedNeuronSet(
    base_neuron_set=ref_a, combined_with=[(ref_b, SetOperation.UNION)]
)
combined_union.set_block_name("union_ab")
print(f"Union: {combined_union.get_neuron_ids(circuit)}")
print(f"  Populations: {combined_union.get_populations(circuit)}")
print(f"  Neuron IDs: {combined_union.get_neuron_ids(circuit)}")
print(f"  Node set def: {combined_union.get_node_set_definition(circuit)}")

In [ ]:
# Intersection
combined_intersect = BiophysicalCombinedNeuronSet(
    base_neuron_set=ref_a, combined_with=[(ref_b, SetOperation.INTERSECT)]
)
combined_intersect.set_block_name("intersect_ab")
print(f"Intersection: {combined_intersect.get_neuron_ids(circuit)}")
print(f"  Populations: {combined_intersect.get_populations(circuit)}")
print(f"  Neuron IDs: {combined_intersect.get_neuron_ids(circuit)}")
print(f"  Node set def: {combined_intersect.get_node_set_definition(circuit)}")

In [ ]:
# Difference (A - B)
combined_diff = BiophysicalCombinedNeuronSet(
    base_neuron_set=ref_a, combined_with=[(ref_b, SetOperation.DIFF)]
)
combined_diff.set_block_name("diff_ab")
print(f"Difference (A - B): {combined_diff.get_neuron_ids(circuit)}")
print(f"  Populations: {combined_diff.get_populations(circuit)}")
print(f"  Neuron IDs: {combined_diff.get_neuron_ids(circuit)}")
print(f"  Node set def: {combined_diff.get_node_set_definition(circuit)}")

#### (b) Recursive cycle detection in combined neuron sets

In [ ]:
# Create a recursive cycle: combined_x references combined_y, which references combined_x
combined_x = BiophysicalCombinedNeuronSet(
    base_neuron_set=ref_a, combined_with=[(ref_b, SetOperation.UNION)]
)
combined_x.set_block_name("combined_x")

combined_y = BiophysicalCombinedNeuronSet(
    base_neuron_set=ref_a, combined_with=[(ref_b, SetOperation.UNION)]
)
combined_y.set_block_name("combined_y")

# Now create the cycle: x references y, y references x
ref_x = BiophysicalNeuronSetReference(block_dict_name="neuron_sets", block_name="combined_x")
ref_x.block = combined_x
ref_y = BiophysicalNeuronSetReference(block_dict_name="neuron_sets", block_name="combined_y")
ref_y.block = combined_y

# Rewire to create the cycle
combined_x.combined_with = [(ref_y, SetOperation.UNION)]  # x combines with y
combined_y.combined_with = [(ref_x, SetOperation.UNION)]  # y combines with x (CYCLE!)

# This should raise a ValueError about recursive loop
try:
    combined_x.get_neuron_ids(circuit)
except ValueError as e:
    print(f"Caught expected error: {e}")

#### (c) BiophysicalCombinedNeuronSet — union kept symbolic

In [ ]:
# Create two base neuron sets to combine
nset_a = BiophysicalPopulationPredefinedNeuronSet(node_set="Layer3", population="S1nonbarrel_neurons")
nset_a.set_block_name("nset_a")

nset_b = BiophysicalPopulationPredefinedNeuronSet(node_set="Layer6", population="S1nonbarrel_neurons")
nset_b.set_block_name("nset_b")

# Wrap in BlockReferences (as required by the combined neuron set fields)
ref_a = BiophysicalNeuronSetReference(block_dict_name="neuron_sets", block_name="nset_a")
ref_a.block = nset_a
ref_b = BiophysicalNeuronSetReference(block_dict_name="neuron_sets", block_name="nset_b")
ref_b.block = nset_b

print(f"Set A ({nset_a.node_set}): {nset_a.get_node_set_definition(circuit)}")
print(f"Set B ({nset_b.node_set}): {nset_b.get_node_set_definition(circuit)}")

combined_union = BiophysicalCombinedNeuronSet(
    base_neuron_set=ref_a, combined_with=[(ref_b, SetOperation.UNION)]
)
combined_union.set_block_name("union_ab")
print("Union:")
print(f"  Node set def: {combined_union.get_node_set_definition(circuit)}")

# Add to circuit and resolve
sonata_circuit = circuit.sonata_circuit
nset_name = combined_union.add_node_set_definition_to_sonata_circuit(circuit, sonata_circuit)
ntab = sonata_circuit.nodes["S1nonbarrel_neurons"].get(nset_name)
ntab.head()

#### (d) VirtualCombinedNeuronSet — union across multiple virtual populations

In [ ]:
# Create two virtual neuron sets from different populations
nset_pom = VirtualPopulationNeuronSet(population="POm")
nset_pom.set_block_name("nset_pom")

nset_vpm = VirtualPopulationNeuronSet(population="VPM")
nset_vpm.set_block_name("nset_vpm")

# Wrap in VirtualNeuronSetReferences
ref_pom = VirtualNeuronSetReference(block_dict_name="neuron_sets", block_name="nset_pom")
ref_pom.block = nset_pom
ref_vpm = VirtualNeuronSetReference(block_dict_name="neuron_sets", block_name="nset_vpm")
ref_vpm.block = nset_vpm

# Union across two virtual populations
combined_virtual = VirtualCombinedNeuronSet(
    base_neuron_set=ref_pom, combined_with=[(ref_vpm, SetOperation.UNION)]
)
combined_virtual.set_block_name("virtual_union")
print(f"Virtual union: {combined_virtual.get_neuron_ids(circuit)}")
print(f"  Populations: {combined_virtual.get_populations(circuit)}")
print(f"  Node set def: {combined_virtual.get_node_set_definition(circuit)}")

#### (e) BiophysicalCombinedNeuronSet — combining 3 neuron sets with different operations

In [ ]:
# Create three base neuron sets
nset_1 = BiophysicalPopulationPredefinedNeuronSet(node_set="All", population="S1nonbarrel_neurons", sample_percentage=60, sample_seed=0)
nset_1.set_block_name("nset_1")

nset_2 = BiophysicalPopulationPredefinedNeuronSet(node_set="All", population="S1nonbarrel_neurons", sample_percentage=60, sample_seed=1)
nset_2.set_block_name("nset_2")

nset_3 = BiophysicalPopulationPredefinedNeuronSet(node_set="All", population="S1nonbarrel_neurons", sample_percentage=40, sample_seed=2)
nset_3.set_block_name("nset_3")

# Wrap in BlockReferences
ref_1 = BiophysicalNeuronSetReference(block_dict_name="neuron_sets", block_name="nset_1")
ref_1.block = nset_1
ref_2 = BiophysicalNeuronSetReference(block_dict_name="neuron_sets", block_name="nset_2")
ref_2.block = nset_2
ref_3 = BiophysicalNeuronSetReference(block_dict_name="neuron_sets", block_name="nset_3")
ref_3.block = nset_3

print(f"Set 1 (60%, seed 0): {nset_1.get_neuron_ids(circuit)}")
print(f"Set 2 (60%, seed 1): {nset_2.get_neuron_ids(circuit)}")
print(f"Set 3 (40%, seed 2): {nset_3.get_neuron_ids(circuit)}")

# Combine: (Set 1 UNION Set 2) DIFF Set 3
# Operations are applied sequentially from top to bottom
combined_multi = BiophysicalCombinedNeuronSet(
    base_neuron_set=ref_1,
    combined_with=[
        (ref_2, SetOperation.UNION),
        (ref_3, SetOperation.DIFF),
    ],
)
combined_multi.set_block_name("multi_combined")
print(f"\n(Set 1 UNION Set 2) DIFF Set 3: {combined_multi.get_neuron_ids(circuit)}")
print(f"  Populations: {combined_multi.get_populations(circuit)}")
print(f"  Node set def: {combined_multi.get_node_set_definition(circuit)}")